# BrightCart Daily Sales Pipeline

The **same steps as the Build lab**, now applied to the real daily files.

| Lab | Here |
|---|---|
| Data typed inside the code with `pd.DataFrame(...)` | Data **arrives as files** in `data/raw/<RUN_DATE>/` |
| One rule per exercise | All rules combined, because real data has many problems at once |
| Results printed on screen | Results **saved** to `cleaned`, `curated` and `audit` |

Each section says which lab question it comes from.

**Business rule:** an order counts as completed revenue only when its status is `COMPLETED`, quantity and price are valid, and duplicate order IDs have been removed.

---
## 0. Setup

All paths are built from the project folder with `pathlib`, so the notebook works on Mac and Windows, and on any computer.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json

import pandas as pd
import numpy as np

# The notebook lives in /notebooks, so the project root is one level up
PROJECT_ROOT = Path.cwd().parent

# The ONLY thing that changes from one day to the next.
# Later, Airflow will give us this value automatically for each daily run.
RUN_DATE = "20261105"

RAW_DIR = PROJECT_ROOT / "data" / "raw" / RUN_DATE
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned" / RUN_DATE
CURATED_DIR = PROJECT_ROOT / "data" / "curated" / RUN_DATE
AUDIT_FILE = PROJECT_ROOT / "data" / "audit" / f"{RUN_DATE}.json"

# Business constants, written once and in one place
VALID_STATUSES = ["COMPLETED", "CANCELLED", "PENDING", "REFUNDED"]
REVENUE_STATUS = "COMPLETED"

print("Processing day:", RUN_DATE)
print("Reading from:  ", RAW_DIR)

Processing day: 20261105
Reading from:   /Users/anyelle/Classes - IH/Week 07 - MiniProject/data/raw/20261105


---
## 1. Extract: read the raw files

From lab **Q11** (preserve raw data).

We read every column as **text** (`dtype=str`). CSV files do not guarantee types: a quantity written as `two` or a price written as `£24.99` would silently change the whole column type. Reading as text first and converting on purpose keeps us in control.

`raw_customers` and `raw_orders` are **never modified**. All cleaning happens on copies.

In [2]:
# Fail early with a clear message if a file is missing (this becomes the
# check_input_files task in Airflow)
for name in ["customers.csv", "orders.csv"]:
    path = RAW_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path}")

raw_customers = pd.read_csv(RAW_DIR / "customers.csv", dtype=str)
raw_orders = pd.read_csv(RAW_DIR / "orders.csv", dtype=str)

print("customers:", raw_customers.shape)
print("orders:   ", raw_orders.shape)

customers: (46, 4)
orders:    (61, 7)


---
## 2. Inspect before transforming

From the lesson, section 4, and lab **Q14** (data quality tests). We only **look** here, nothing changes yet.

In [3]:
print("Missing values in orders:")
print(raw_orders.isna().sum())

print("\nHow status is written:")
print(raw_orders["status"].value_counts(dropna=False))

print("\nRepeated order_id:", raw_orders["order_id"].duplicated().sum())
print("Repeated customer_id:", raw_customers["customer_id"].duplicated().sum())

Missing values in orders:
order_id          0
customer_id       1
order_date        0
product_name      0
quantity          0
unit_price_gbp    2
status            0
dtype: int64

How status is written:
status
COMPLETED      27
Completed       8
COMPLETED       7
 completed      4
pending         3
completed       3
refunded        3
PENDING         2
REFUNDED        1
SHIPPD          1
CANCELLED       1
cancelled       1
Name: count, dtype: int64

Repeated order_id: 3
Repeated customer_id: 1


In [4]:
# Values that will NOT convert to numbers. These need a decision: fix or reject.
for col in ["quantity", "unit_price_gbp"]:
    not_numeric = raw_orders[col].notna() & pd.to_numeric(raw_orders[col], errors="coerce").isna()
    print(f"{col} values that are not numbers:", raw_orders.loc[not_numeric, col].tolist())

quantity values that are not numbers: ['two']
unit_price_gbp values that are not numbers: ['£29.00']


**What the inspection tells us:**

* `status` is written in several ways (`COMPLETED`, `completed`, ` completed `...) and one value, `SHIPPD`, is not a real status.
* `unit_price_gbp` has a value with `£`. That is safe to **fix**.
* `quantity` has the word `two`. We cannot be sure of the meaning, so we **reject** it.
* There are repeated `order_id` and one repeated `customer_id`.

These observations become the rules below.

---
## 3. Validate customers

From lab **Q1** (required fields), **Q6** (fix safe values), **Q18** (duplicate keys break joins) and **Q19** (rejection reason).

Rules:

* `customer_id` and `customer_name` are required. `email` is optional.
* Trim spaces in names and cities, and write cities in title case.
* A repeated `customer_id` keeps the first row and rejects the others.

In [5]:
customers = raw_customers.copy()

# FIX first (lab Q6). An empty name after trimming counts as missing.
customers["customer_name"] = customers["customer_name"].str.strip().replace("", np.nan)
customers["city"] = customers["city"].str.strip().str.title()
customers["customer_id"] = pd.to_numeric(customers["customer_id"], errors="coerce").astype("Int64")

# REJECT with a reason (lab Q19). np.select uses the FIRST condition that is True.
conditions = [
    customers["customer_id"].isna(),
    customers["customer_name"].isna(),
    customers["customer_id"].duplicated(keep="first"),
]
reasons = ["missing_customer_id", "missing_customer_name", "duplicate_customer_id"]
customers["rejection_reason"] = np.select(conditions, reasons, default="valid")

valid_customers = customers[customers["rejection_reason"] == "valid"].drop(columns="rejection_reason")
rejected_customers = customers[customers["rejection_reason"] != "valid"]

print("valid:", len(valid_customers), "| rejected:", len(rejected_customers))
print("reconciles:", len(customers) == len(valid_customers) + len(rejected_customers))
rejected_customers

valid: 44 | rejected: 2
reconciles: True


,customer_id,customer_name,city,email,rejection_reason
20,12,Ruby Turner,Leeds,ruby.turner@example.co.uk,duplicate_customer_id
37,37,NaN,Liverpool,isabella.quinn@example.co.uk,missing_customer_name


---
## 4. Clean orders: safe fixes

From lab **Q6** and the lesson, sections 5 and 6.

Fixes we are sure about: standardise status, remove `£`, trim product names, accept dates written as day/month/year. After this, we convert to numbers with `errors="coerce"`, so anything that still cannot be converted becomes missing and is caught by the rules.

In [6]:
orders = raw_orders.copy()

# Standardise text ONCE, so every later rule compares a single value
orders["status"] = orders["status"].str.strip().str.upper()
orders["product_name"] = orders["product_name"].str.strip()

# Safe fix: "£24.99" means 24.99
orders["unit_price_gbp"] = orders["unit_price_gbp"].str.replace("£", "", regex=False)

# Convert to numbers. Anything invalid ("two", empty) becomes NaN
orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")
orders["unit_price_gbp"] = pd.to_numeric(orders["unit_price_gbp"], errors="coerce")
orders["customer_id"] = pd.to_numeric(orders["customer_id"], errors="coerce").astype("Int64")

# Dates: accept ISO (2026-11-05) and UK format (05/11/2026)
iso_dates = pd.to_datetime(orders["order_date"], format="%Y-%m-%d", errors="coerce")
uk_dates = pd.to_datetime(orders["order_date"], format="%d/%m/%Y", errors="coerce")
orders["order_date"] = iso_dates.fillna(uk_dates)

orders.head()

,order_id,customer_id,order_date,product_name,quantity,unit_price_gbp,status
0,10128,36,2026-11-05,USB C Cable,2.0,9.99,REFUNDED
1,10152,30,2026-11-05,Wireless Mouse,2.0,24.99,COMPLETED
2,10146,12,2026-11-05,Noise Cancelling Headphones,2.0,149.00,COMPLETED
3,10165,35,2026-11-05,Monitor Arm,2.0,64.00,PENDING
4,10133,16,2026-11-05,Notebook Pack,1.0,7.50,COMPLETED


---
## 5. Validate orders with a rejection reason

From lab **Q2** (reject invalid rows), **Q14** (duplicates and missing IDs) and **Q19** (rejection reason).

Every rejected row keeps **one reason**. When a row has several problems, the first reason in this list wins:

1. `duplicate_order_id`: second time the same order appears
2. `missing_customer_id`
3. `unknown_customer`: customer not in today's valid customers
4. `invalid_quantity`: zero, negative or not a number
5. `missing_price`
6. `invalid_price`: zero or negative
7. `invalid_status`: not one of the accepted statuses
8. `invalid_date`

Duplicates are **not** dropped silently with `drop_duplicates()`. They go to the rejected output, so the audit can count them.

In [7]:
known_customer_ids = set(valid_customers["customer_id"])

conditions = [
    orders["order_id"].duplicated(keep="first"),
    orders["customer_id"].isna(),
    ~orders["customer_id"].isin(known_customer_ids),
    orders["quantity"].isna() | (orders["quantity"] <= 0),
    orders["unit_price_gbp"].isna(),
    orders["unit_price_gbp"] <= 0,
    ~orders["status"].isin(VALID_STATUSES),
    orders["order_date"].isna(),
]
reasons = [
    "duplicate_order_id",
    "missing_customer_id",
    "unknown_customer",
    "invalid_quantity",
    "missing_price",
    "invalid_price",
    "invalid_status",
    "invalid_date",
]
orders["rejection_reason"] = np.select(conditions, reasons, default="valid")

valid_orders = orders[orders["rejection_reason"] == "valid"].drop(columns="rejection_reason").copy()
rejected_orders = orders[orders["rejection_reason"] != "valid"].copy()

print("valid:", len(valid_orders), "| rejected:", len(rejected_orders))
print("reconciles:", len(orders) == len(valid_orders) + len(rejected_orders))
print()
print(rejected_orders["rejection_reason"].value_counts())

valid: 48 | rejected: 13
reconciles: True

rejection_reason
invalid_quantity       3
duplicate_order_id     3
missing_price          2
unknown_customer       2
invalid_status         1
invalid_price          1
missing_customer_id    1
Name: count, dtype: int64


**Rejection in cascade:** on 03/11, customer 37 is rejected because the name is missing. His orders then become `unknown_customer`. Change `RUN_DATE` to `"20261103"` to see it.

---
## 6. Join customers to orders

From lab **Q7** (safe many to one join) and **Q18** (dangerous join).

`validate="many_to_one"` makes the join fail loudly if a customer appears twice. We also check that the join did not add or remove rows.

In [8]:
orders_enriched = valid_orders.merge(
    valid_customers[["customer_id", "customer_name", "city"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

# Safety checks: same number of rows, and every valid order found its customer
assert len(orders_enriched) == len(valid_orders), "The join changed the number of orders"
assert orders_enriched["city"].notna().all(), "Some valid orders have no city"

print("orders after join:", len(orders_enriched))

orders after join: 48


---
## 7. Derived column: revenue per order

From lab **Q9**. Calculated only now, after quantity and price are validated.

In [9]:
orders_enriched["line_revenue_gbp"] = (
    orders_enriched["quantity"] * orders_enriched["unit_price_gbp"]
).round(2)

orders_enriched[["order_id", "city", "quantity", "unit_price_gbp", "status", "line_revenue_gbp"]].head()

,order_id,city,quantity,unit_price_gbp,status,line_revenue_gbp
0,10128,Leeds,2.0,9.99,REFUNDED,19.98
1,10152,Glasgow,2.0,24.99,COMPLETED,49.98
2,10146,Leeds,2.0,149.00,COMPLETED,298.00
3,10165,Manchester,2.0,64.00,PENDING,128.00
4,10133,Leeds,1.0,7.50,COMPLETED,7.50


---
## 8. Curated outputs: the daily report

From lab **Q10** (curated report) and the lesson, sections 1 and 10.

The operations team asked for: completed revenue, revenue by city, completed order count, highest value completed order, and records rejected. Cancelled, pending and refunded orders stay in the cleaned data but are excluded here.

In [10]:
completed = orders_enriched[orders_enriched["status"] == REVENUE_STATUS]

# Revenue by city
city_sales = (
    completed
    .groupby("city", as_index=False)
    .agg(
        completed_orders=("order_id", "count"),
        completed_revenue_gbp=("line_revenue_gbp", "sum"),
    )
    .sort_values("completed_revenue_gbp", ascending=False)
)
city_sales["completed_revenue_gbp"] = city_sales["completed_revenue_gbp"].round(2)
city_sales.insert(0, "run_date", RUN_DATE)

# Highest value completed order
top_order = completed.loc[completed["line_revenue_gbp"].idxmax()]

# Daily summary: one row with the headline numbers
daily_summary = pd.DataFrame([{
    "run_date": RUN_DATE,
    "completed_orders": len(completed),
    "completed_revenue_gbp": round(completed["line_revenue_gbp"].sum(), 2),
    "top_order_id": top_order["order_id"],
    "top_order_revenue_gbp": top_order["line_revenue_gbp"],
    "rejected_orders": len(rejected_orders),
}])

print(daily_summary.T)
city_sales

                              0
run_date               20261105
completed_orders             39
completed_revenue_gbp   4011.28
top_order_id              10143
top_order_revenue_gbp     445.0
rejected_orders              13


,run_date,city,completed_orders,completed_revenue_gbp
5,20261105,London,8,1227.98
3,20261105,Leeds,9,857.45
0,20261105,Bristol,8,584.44
7,20261105,Newcastle,2,495.00
2,20261105,Glasgow,5,310.47
6,20261105,Manchester,3,307.98
1,20261105,Edinburgh,3,198.96
4,20261105,Liverpool,1,29.00


---
## 9. Save the outputs

From the lesson, sections 11 and 12, and **idempotent reruns** (section 20).

Each output is **replaced** on every run (`to_csv` overwrites the file). Running this notebook twice for the same day gives exactly the same files, never duplicated rows. The raw folder is never written to.

| Zone | Folder | Files |
|---|---|---|
| Raw | `data/raw/20261105/` | `customers.csv`, `orders.csv` (untouched) |
| Cleaned | `data/cleaned/20261105/` | valid and rejected customers and orders |
| Curated | `data/curated/20261105/` | `city_sales.csv`, `daily_summary.csv` |
| Audit | `data/audit/` | `20261105.json` |

In [11]:
for folder in [CLEANED_DIR, CURATED_DIR, AUDIT_FILE.parent]:
    folder.mkdir(parents=True, exist_ok=True)

# Cleaned zone: row level detail, valid AND rejected (nothing dropped silently)
valid_customers.to_csv(CLEANED_DIR / "customers_valid.csv", index=False)
rejected_customers.to_csv(CLEANED_DIR / "customers_rejected.csv", index=False)
orders_enriched.to_csv(CLEANED_DIR / "orders_valid.csv", index=False)
rejected_orders.to_csv(CLEANED_DIR / "orders_rejected.csv", index=False)

# Curated zone: shaped for the business question
city_sales.to_csv(CURATED_DIR / "city_sales.csv", index=False)
daily_summary.to_csv(CURATED_DIR / "daily_summary.csv", index=False)

print("Saved:")
for path in sorted(CLEANED_DIR.iterdir()) + sorted(CURATED_DIR.iterdir()):
    print(" ", path.relative_to(PROJECT_ROOT))

Saved:
  data/cleaned/20261105/customers_rejected.csv
  data/cleaned/20261105/customers_valid.csv
  data/cleaned/20261105/orders_rejected.csv
  data/cleaned/20261105/orders_valid.csv
  data/curated/20261105/city_sales.csv
  data/curated/20261105/daily_summary.csv


---
## 10. Audit record

From lab **Q13** (audit record), **Q15** (reconciliation) and **Q20** (better audit fields).

The audit is the evidence that this run can be trusted: what arrived, what happened to every row, and the business result.

In [12]:
audit = {
    "run_date": RUN_DATE,
    "run_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "customers": {
        "source": len(raw_customers),
        "valid": len(valid_customers),
        "rejected": len(rejected_customers),
        "rejected_by_reason": rejected_customers["rejection_reason"].value_counts().to_dict(),
    },
    "orders": {
        "source": len(raw_orders),
        "valid": len(valid_orders),
        "rejected": len(rejected_orders),
        "rejected_by_reason": rejected_orders["rejection_reason"].value_counts().to_dict(),
    },
    "output": {
        "completed_orders": int(len(completed)),
        "completed_revenue_gbp": float(daily_summary.loc[0, "completed_revenue_gbp"]),
        "cities_in_report": int(len(city_sales)),
    },
}
audit["customers"]["reconciles"] = audit["customers"]["source"] == audit["customers"]["valid"] + audit["customers"]["rejected"]
audit["orders"]["reconciles"] = audit["orders"]["source"] == audit["orders"]["valid"] + audit["orders"]["rejected"]

print(json.dumps(audit, indent=2))

{
  "run_date": "20261105",
  "run_at_utc": "2026-11-05T15:46:18+00:00",
  "customers": {
    "source": 46,
    "valid": 44,
    "rejected": 2,
    "rejected_by_reason": {
      "duplicate_customer_id": 1,
      "missing_customer_name": 1
    },
    "reconciles": true
  },
  "orders": {
    "source": 61,
    "valid": 48,
    "rejected": 13,
    "rejected_by_reason": {
      "invalid_quantity": 3,
      "duplicate_order_id": 3,
      "missing_price": 2,
      "unknown_customer": 2,
      "invalid_status": 1,
      "invalid_price": 1,
      "missing_customer_id": 1
    },
    "reconciles": true
  },
  "output": {
    "completed_orders": 39,
    "completed_revenue_gbp": 4011.28,
    "cities_in_report": 8
  }
}


---
## 11. Quality checks before declaring success

From lab **Q14** and **Q15**, and the lesson, section 19.

A pipeline is not successful just because the code finished. These checks are the **publishing gate**: in Airflow they will become the `run_quality_check` task, and a failure will stop the run.

In [13]:
checks = {
    "order counts reconcile": audit["orders"]["reconciles"],
    "customer counts reconcile": audit["customers"]["reconciles"],
    "no duplicate order_id in valid output": orders_enriched["order_id"].is_unique,
    "no missing quantity or price in valid output": orders_enriched[["quantity", "unit_price_gbp"]].notna().all().all(),
    "revenue: cleaned = city report": round(completed["line_revenue_gbp"].sum(), 2) == round(city_sales["completed_revenue_gbp"].sum(), 2),
    "revenue: city report = audit": round(city_sales["completed_revenue_gbp"].sum(), 2) == audit["output"]["completed_revenue_gbp"],
    "output files exist": all((CURATED_DIR / f).exists() for f in ["city_sales.csv", "daily_summary.csv"]),
}

for name, passed in checks.items():
    print("PASS" if passed else "FAIL", " ", name)

audit["quality_checks"] = {name: bool(passed) for name, passed in checks.items()}
audit["status"] = "success" if all(checks.values()) else "failed"

# Replaced on every run, so a rerun never leaves an old audit behind
AUDIT_FILE.write_text(json.dumps(audit, indent=2))
print("\nRun status:", audit["status"].upper())
print("Audit saved to:", AUDIT_FILE.relative_to(PROJECT_ROOT))

PASS   order counts reconcile
PASS   customer counts reconcile
PASS   no duplicate order_id in valid output
PASS   no missing quantity or price in valid output
PASS   revenue: cleaned = city report
PASS   revenue: city report = audit
PASS   output files exist

Run status: SUCCESS
Audit saved to: data/audit/20261105.json


---
## Try it

1. Change `RUN_DATE` in section 0 to `"20261103"` or `"20261104"` and **Run All**. A new set of folders appears for that day.
2. Run the same day twice. The files are replaced and the numbers do not change: the pipeline is **idempotent**.
3. Rename a raw file and run again. Section 1 stops with a clear `FileNotFoundError`.

## From notebook to project

Each section becomes one function in `scripts/pipeline_tasks.py`, and later one Airflow task:

| Notebook section | Function | Airflow task |
|---|---|---|
| 1. Extract (file check) | `check_input_files()` | `check_input_files` |
| 3 to 7. Validate, join, derive, save cleaned | `clean_orders()` | `clean_orders` |
| 8 to 10. Curated outputs and audit | `build_city_summary()` | `build_city_summary` |
| 11. Quality checks | `run_quality_check()` | `run_quality_check` |